# Lab Exercise: Loading Data Defensively in Pandas
This notebook guides you through the process of auditing and parsing messy dataset inputs.

You will see how Pandas inferencing rules routinely hide numerical columns as generic objects, how to run statistical data-audits, and how to configure custom missing-value markers defensively.

In [ ]:
import numpy as np
import pandas as pd

# Let's write a messy CSV file to run our experiments on.
with open('messy.csv', 'w', encoding='utf-8') as f:
    f.write('ID,Name,Age,Salary,Region,Status\n' 
            '1,Alice,25,"55,000",North,Active\n' 
            '2,Bob,?,?,\"North, East\",Active\n' 
            '3,Charlie,30,\"61,000\",South,Active\n' 
            '4,Dave,-1,45000,East,Active\n' 
            '5,Alice,25,\"55,000\",North,Active\n' 
            '6,Eve,40,999999,West,Active\n')

print("✓ Mock 'messy.csv' compiled on disk successfully.")

### 1. The Naive Load
Let's load the file using the default settings of `pd.read_csv`. We will print column dtypes and see how Pandas inferencing handled Age and Salary.

In [ ]:
naive = pd.read_csv('messy.csv')
print('1. NAIVE read_csv — dtypes it inferred')
print(naive.dtypes.to_string())
print(f'\n   missing values reported: {int(naive.isna().sum().sum())}')
print("   Observe: Age and Salary are 'object' columns — they will not do arithmetic!")

### 2. Default NA Values
Let's explore the standard string markers Pandas automatically treats as missing.

In [ ]:
print(f'pandas treats {len(pd._libs.parsers.STR_NA_VALUES)} strings as missing by default:')
print('  ' + ', '.join(sorted(repr(s) for s in pd._libs.parsers.STR_NA_VALUES)))
print("\nNote what is NOT in that list: '?', '-', 'missing', -1, or 999999.")

### 3. The Data Audit Function
We define a robust pre-flight audit function to search for duplicate entries, constant columns, and numeric columns masked as objects.

In [ ]:
def audit(df, name):
    print(f'\nAUDIT — {name}: {df.shape[0]} rows x {df.shape[1]} columns')
    dups = int(df.duplicated().sum())
    print(f'   duplicate rows:   {dups}' + ('   <- investigate duplicates' if dups else ''))
    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print(f'   constant columns: {const if const else "none"}' + ('   <- no predictive value' if const else ''))
    print(f'\n   {"column":<12}{"dtype":>9}{"missing":>9}{"unique":>8}   {"note":<28}')
    for c in df.columns:
        note = ''
        if df[c].dtype == object:
            coerced = pd.to_numeric(df[c], errors='coerce')
            if coerced.notna().sum() > 0.5 * df[c].notna().sum():
                note = 'LOOKS NUMERIC but is object'
        elif int((df[c] < 0).sum()):
            note = f'{int((df[c] < 0).sum())} negative value(s)'
        print(f'   {c:<12}{str(df[c].dtype):>9}{int(df[c].isna().sum()):>9}{df[c].nunique():>8}   {note:<28}')

audit(naive, 'naive read')

### 4. Corrected Defensive Loading
Now we declare our custom domain-knowledge missingness tags (`na_values`) and thousand separators explicitly.

In [ ]:
df = pd.read_csv('messy.csv',
                 na_values=['?', '-1', '999999'],  # Custom domain knowledge
                 thousands=',')

audit(df, 'corrected read')

### 5. Indexing: loc vs. iloc
We trace loc (inclusive) vs iloc (exclusive) indices, showing how sorting breaks row synchronization.

In [ ]:
print('iloc (exclusive) indexing of range [1:3]:\n')
print(df.iloc[1:3])
print('\nloc (inclusive) indexing of index keys [1:3]:\n')
print(df.loc[1:3])

### Clean Up
Clean up the temporary `.csv` file.

In [ ]:
import os
if os.path.exists('messy.csv'):
    os.remove('messy.csv')
print('✓ Local files cleaned up.')